# Step 2b · The direct effect of outages on local activity

**Question**: the longer a county is without power, the fewer visits do its places receive? And does the relationship hold once the wind, rain and cold the county went through are accounted for?

**Approach**: each row is "event × county × week". The outcome is Step 1b's `visits_in_chg` (visits to places in the county relative to the baseline, in log points). The explanatory variable is the week's hours without power per customer, in five bands:

| Band | Hours without power per customer this week | Roughly |
| --- | --- | --- |
| Reference | Under 6 hours | No substantial outage |
| 6–24 | 6 to 24 hours | Less than a day |
| 24–72 | 1 to 3 days | |
| 72–120 | 3 to 5 days | |
| 120+ | More than 5 days | Nearly the whole week |

Every model has **event × week fixed effects**: all counties in the same event and week share one intercept, which absorbs the shared ups and downs of holidays, seasons and the pandemic. A coefficient therefore means: **within the same event and week, how many log points more (or fewer) visits a county without power for this long had than a county without a substantial outage.** The outcome is already a change from the county's own baseline, so fixed differences between counties are removed and no county fixed effects are added.

Controls for the storm itself (from Step 2a, all in bands to allow non-linear effects):

- **This week**: highest gust (county mean), total precipitation, frozen precipitation
- **Cumulative**: the highest gust and total precipitation from week −1 up to this week. Outages often last beyond the wind, and so does damage to homes and roads; the cumulative variables are there to absorb that

**Sample**: the 7 events other than Uri; counties with `flag_base`, `flag_prior` or `low_sample` are left out. Uri is estimated separately, for reference only (in Step 1b its own comparison group fell by 31 log points). Standard errors are clustered by county (a county appears in several events).

**Structure**: A data → B the decision rule, fixed in advance → C main results and robustness → D other outcomes → E event by event → F event study → G three figures → H recovery timing → I checklist. No Earth Engine is needed; it runs in a few minutes.

In [ ]:
# 1. Environment, folders, chart style
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display
try:
    import statsmodels.formula.api as smf
except ImportError:                     # Colab usually has statsmodels; install it if not
    import subprocess, sys
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "statsmodels"], check=True)
    import statsmodels.formula.api as smf

try:
    from google.colab import drive
    drive.mount("/content/drive")
    BASE = "/content/drive/MyDrive"
except ImportError:
    BASE = "."
OUT = f"{BASE}/outage_mobility"
for f in ["step1b_events.csv", "step1b_outage_event.csv", "step1b_panel_weekly.csv.gz", "step2a_hazard_weekly.csv"]:
    if not os.path.exists(f"{OUT}/{f}"):
        raise FileNotFoundError(f"{OUT}/{f} is missing; run Steps 1b and 2a first.")

def tidy(df, n=1):
    # round numeric columns only, to keep tables readable
    return df.round({c: n for c in df.select_dtypes("number").columns})

INK, MUTED, GRID, SURFACE, FAINT = "#0b0b0b", "#52514e", "#e6e5e1", "#fcfcfb", "#b9b8b2"
C_DARK, C_MID, C_LIGHT, C_GREY = "#104281", "#2a78d6", "#86b6ef", "#7a7974"
plt.rcParams.update({
    "figure.facecolor": SURFACE, "axes.facecolor": SURFACE, "savefig.facecolor": SURFACE,
    "axes.edgecolor": GRID, "axes.grid": True, "grid.color": GRID, "grid.linewidth": 0.8,
    "axes.spines.top": False, "axes.spines.right": False, "axes.axisbelow": True,
    "text.color": INK, "axes.labelcolor": MUTED, "xtick.color": MUTED, "ytick.color": MUTED,
    "font.size": 10, "axes.titlesize": 11, "axes.titleweight": "bold", "axes.titlelocation": "left",
    "legend.frameon": False, "figure.dpi": 110, "savefig.dpi": 200, "savefig.bbox": "tight",
})

In [ ]:
# 2. Parameters
HOLDOUT = ["uri2021"]                          # kept out of the main sample, estimated separately
OUT_EDGES  = [0, 6, 24, 72, 120, np.inf]       # bands of the week's hours without power per customer
OUT_LEVELS = ["h000", "h006", "h024", "h072", "h120"]
OUT_NAMES  = {"h000": "<6 h", "h006": "6–24 h", "h024": "24–72 h", "h072": "72–120 h", "h120": "120+ h", "h072p": "72+ h"}
GUST_EDGES = [-np.inf, 17.5, 25, 33, np.inf]   # gust (m/s): below tropical storm, tropical storm, strong tropical storm, hurricane
RAIN_EDGES = [-np.inf, 25, 75, 150, np.inf]    # weekly precipitation (mm)
COLD_EDGES = [-np.inf, 5, 25, np.inf]          # weekly frozen precipitation (mm)
RAINCUM_EDGES = [-np.inf, 50, 150, 300, np.inf]  # cumulative precipitation (mm)
CUM_FROM = -1                                  # cumulative wind and rain are counted from week −1
MIN_FIT = 3                                    # an event needs at least this many county-weeks at 72+ h to be estimated on its own
MIN_TREATED = 5                                # and at least this many to count in part 2 of the decision rule
ES_REF = -2                                    # reference week of the event study
RECOVERED = -10                                # visits within 10 log points of the counties without a substantial outage count as recovered

## A. Data

Step 1b's weekly panel and event labels are merged with Step 2a's hazard table, and the continuous variables are put into bands. The table below counts county-weeks in each outage band for each event of the main sample: the cells at 72+ h are few and concentrated in a few events, which decides which conclusions below can stand.

In [ ]:
# 3. Merge, bands, cumulative hazard
EVENTS = pd.read_csv(f"{OUT}/step1b_events.csv", index_col=0)
for c in ["t0", "day0", "week0"]:
    EVENTS[c] = pd.to_datetime(EVENTS[c])
W  = pd.read_csv(f"{OUT}/step1b_panel_weekly.csv.gz", dtype={"fips": str})
L  = pd.read_csv(f"{OUT}/step1b_outage_event.csv", dtype={"fips": str})
HZ = pd.read_csv(f"{OUT}/step2a_hazard_weekly.csv", dtype={"fips": str})

HZ = HZ.sort_values(["event", "fips", "rel_week"])
after = HZ["rel_week"] >= CUM_FROM
grp = [HZ["event"], HZ["fips"]]
HZ["gust_cum"] = HZ["gust_mean_ms"].where(after).groupby(grp).cummax().fillna(HZ["gust_mean_ms"])
HZ["rain_cum"] = HZ["rain_mm"].where(after, 0).groupby(grp).cumsum().where(after, HZ["rain_mm"])

D = (W.merge(L[["event", "fips", "name", "group", "hours_above", "restored_time", "flag_base", "flag_prior", "flag_late"]], on=["event", "fips"])
      .merge(HZ[["event", "fips", "rel_week", "gust_mean_ms", "rain_mm", "cold_ppt_mm", "tmin_c", "gust_cum", "rain_cum"]],
             on=["event", "fips", "rel_week"], how="left"))
D["out_bin"]  = pd.cut(D["out_week_hours"], OUT_EDGES, right=False, labels=OUT_LEVELS).astype(str)
D["out_bin4"] = D["out_bin"].replace({"h072": "h072p", "h120": "h072p"})       # 72+ h merged into one band for the event-by-event estimates
D["gust_bin"]    = pd.cut(D["gust_mean_ms"], GUST_EDGES, labels=["g0", "g1", "g2", "g3"]).astype(str)
D["gustcum_bin"] = pd.cut(D["gust_cum"], GUST_EDGES, labels=["k0", "k1", "k2", "k3"]).astype(str)
D["rain_bin"]    = pd.cut(D["rain_mm"], RAIN_EDGES, labels=["r0", "r1", "r2", "r3"]).astype(str)
D["raincum_bin"] = pd.cut(D["rain_cum"], RAINCUM_EDGES, labels=["s0", "s1", "s2", "s3"]).astype(str)
D["cold_bin"]    = pd.cut(D["cold_ppt_mm"], COLD_EDGES, labels=["c0", "c1", "c2"]).astype(str)
D["ew"]  = D["event"] + "_" + D["rel_week"].astype(str)                       # event × week
D["esw"] = D["ew"] + "_" + D["state"]                                         # event × state × week
D["kind"] = D["event"].map(EVENTS["kind"])
NEEDED = ["out_week_hours", "gust_mean_ms", "rain_mm", "cold_ppt_mm", "gust_cum", "rain_cum"]
D["keep"] = ~D["flag_base"] & ~D["flag_prior"] & ~D["low_sample"] & D[NEEDED].notna().all(axis=1)
MAIN = D[D["keep"] & ~D["event"].isin(HOLDOUT)].copy()
print(f"Main sample: {len(MAIN):,} county-weeks, {MAIN[['event', 'fips']].drop_duplicates().shape[0]:,} county-events; "
      f"county-weeks dropped for missing hazard data: {int((D[NEEDED[1:]].isna().any(axis=1) & ~D['flag_base'] & ~D['flag_prior']).sum()):,}")
counts = pd.crosstab(MAIN["event"], MAIN["out_bin"]).reindex(columns=OUT_LEVELS, fill_value=0).rename(columns=OUT_NAMES)
counts.loc["all"] = counts.sum()
counts

## B. The decision rule, fixed in advance

This rule is written down before looking at the results, as in the companion outage-duration project.

**If both of the following hold, the conclusion is "when outages approach or exceed three days, local activity falls by more than the storm itself can explain":**

1. In the main specification (current and cumulative wind and rain controlled, event × week fixed effects), **the coefficients for 72–120 h and for 120+ h are both below 0, with 95% confidence intervals that exclude 0**
2. Among events with at least 5 county-weeks at 72+ h, **a majority** have a negative 72+ h coefficient when estimated on their own (the threshold is 5 rather than 10 so that the derecho and Zeta count too: only Laura and Ida reach 10, and both had storm surge)

**Otherwise**, the conclusion is only "in these data, the effect of outages cannot be separated from the effect of the storm itself".

Even if both hold, it cannot be said outright that outages **caused** the drop: storm-surge flooding is not among the controls (rainfall does not capture it), and the hardest-hit counties of Ida and Laura are exactly the ones that flooded. This belongs in the limitations.

In [ ]:
# 4. Estimation helpers
OUT_T  = "C(out_bin, Treatment('h000'))"
OUT4_T = "C(out_bin4, Treatment('h000'))"
NOW_T  = ["C(gust_bin, Treatment('g0'))", "C(rain_bin, Treatment('r0'))", "C(cold_bin, Treatment('c0'))"]
CUM_T  = ["C(gustcum_bin, Treatment('k0'))", "C(raincum_bin, Treatment('s0'))"]

def fit(data, y, terms, fe="ew"):
    # least squares with fixed effects, standard errors clustered by county; returns the model and the rows used
    d = data.dropna(subset=[y])                                            # bands absent from the sample create no empty columns (the banded variables are text)
    formula = f"{y} ~ " + " + ".join(terms + [f"C({fe})"])
    m = smf.ols(formula, data=d).fit(cov_type="cluster", cov_kwds={"groups": pd.factorize(d["fips"])[0]})
    return m, d

def coefs(m, term, names=OUT_NAMES):
    # the coefficients, standard errors and 95% intervals of one banded variable
    ci = m.conf_int()
    rows = []
    for p in m.params.index:
        if p.startswith(term + "[T."):
            lev = p[len(term) + 3:-1]
            rows.append({"band": names.get(lev, lev), "level": lev, "coef": m.params[p], "se": m.bse[p],
                         "lo": ci.loc[p, 0], "hi": ci.loc[p, 1], "p": m.pvalues[p]})
    return pd.DataFrame(rows)

def band_n(d, col, lev):
    return int((d[col] == lev).sum())

## C. Main results and robustness

Five specifications, all with `visits_in_chg` as the outcome:

| Spec | Controls | Fixed effects | Sample |
| --- | --- | --- | --- |
| S1 | Outages only | Event × week | 7 events |
| S2 | + this week's wind and rain | Event × week | 7 events |
| **S3 (main)** | + this week's and cumulative wind and rain | Event × week | 7 events |
| S4 | As S3 | Event × **state** × week | 7 events |
| S5 | As S3 | Event × week | The 5 hurricanes only |

The gap between S1 and S3 is how much of the raw association the storm itself takes up. S4 compares only counties in the same state, which rules out differences between states' pandemic policies, but it also uses storm-touched counties of the same state as comparisons, so its coefficients may be smaller.

In [ ]:
# 5. The five specifications
SPECS = {"S1": ([OUT_T], "ew", None), "S2": ([OUT_T] + NOW_T, "ew", None), "S3": ([OUT_T] + NOW_T + CUM_T, "ew", None),
         "S4": ([OUT_T] + NOW_T + CUM_T, "esw", None), "S5": ([OUT_T] + NOW_T + CUM_T, "ew", "hurricane")}
models, rows = {}, []
for s, (terms, fe, kind) in SPECS.items():
    data = MAIN if kind is None else MAIN[MAIN["kind"] == kind]
    m, used = fit(data, "visits_in_chg", terms, fe)
    models[s] = (m, used)
    c = coefs(m, OUT_T)
    c["spec"], c["n_obs"] = s, int(m.nobs)
    c["n_band"] = [band_n(used, "out_bin", lev) for lev in c["level"]]
    rows.append(c)
main_tab = pd.concat(rows, ignore_index=True)
main_tab.to_csv(f"{OUT}/step2b_main_coefficients.csv", index=False)
show = main_tab.assign(cell=main_tab.apply(lambda r: f"{r['coef']:+.1f} [{r['lo']:+.1f}, {r['hi']:+.1f}]", axis=1))
display(show.pivot(index="band", columns="spec", values="cell").reindex([OUT_NAMES[l] for l in OUT_LEVELS[1:]]))
s1 = main_tab[(main_tab["spec"] == "S1") & (main_tab["level"] == "h120")]["coef"]
s3 = main_tab[(main_tab["spec"] == "S3") & (main_tab["level"] == "h120")]["coef"]
if len(s1) and len(s3):
    print(f"120+ h: S1 {s1.iloc[0]:+.1f}, S3 {s3.iloc[0]:+.1f}; the storm itself takes up {1 - s3.iloc[0] / s1.iloc[0]:.0%} of the raw association")

In [ ]:
# 6. Decision rule, part 1
m3 = main_tab[main_tab["spec"] == "S3"].set_index("level")
rule1_parts = {lev: bool(lev in m3.index and m3.loc[lev, "coef"] < 0 and m3.loc[lev, "hi"] < 0) for lev in ["h072", "h120"]}
rule1 = all(rule1_parts.values())
print("Part 1:", "holds" if rule1 else "does not hold", rule1_parts)

## D. Other outcomes (main specification)

| Outcome | What it reflects |
| --- | --- |
| `visits_in_chg` | Visits to places in the county (main outcome) |
| `visits_in_nonlocal_chg` | Of these, visits from other counties |
| `visits_res_local_chg` | Residents' visits inside their own county |
| `visits_res_chg` | Residents' visits anywhere |
| `revenue_chg`, `merchants_chg` | Small-business revenue and businesses open (where the business is; Womply covers few counties) |
| `spend_chg` | Card spending (where the cardholder lives) |

If `visits_res_chg` falls less than `visits_res_local_chg`, residents are not staying home but going to other counties: the "outflow of activity" that Step 3 tests.

In [ ]:
# 7. Main specification, other outcomes
OUTCOMES = ["visits_in_chg", "visits_in_nonlocal_chg", "visits_res_local_chg", "visits_res_chg", "revenue_chg", "merchants_chg", "spend_chg"]
rows = []
for y in OUTCOMES:
    m, used = fit(MAIN, y, SPECS["S3"][0], "ew")
    c = coefs(m, OUT_T)
    c["outcome"], c["n_obs"], c["n_counties"] = y, int(m.nobs), used["fips"].nunique()
    c["n_band"] = [band_n(used, "out_bin", lev) for lev in c["level"]]
    rows.append(c)
out_tab = pd.concat(rows, ignore_index=True)
out_tab.to_csv(f"{OUT}/step2b_outcomes.csv", index=False)
show = out_tab.assign(cell=out_tab.apply(lambda r: f"{r['coef']:+.1f} [{r['lo']:+.1f}, {r['hi']:+.1f}] (n={r['n_band']})", axis=1))
show.pivot(index="outcome", columns="band", values="cell").reindex(index=OUTCOMES, columns=[OUT_NAMES[l] for l in OUT_LEVELS[1:]])

## E. Event by event

The main specification is run on each event alone, with 72+ h merged into one band (each event has too few county-weeks there to split it in two). Uri appears here too, but does not enter the decision rule.

The key case is the **derecho**: a windstorm with no storm surge and little rain. If its coefficient is clearly negative too, the "it was really the flooding" explanation becomes much weaker.

In [ ]:
# 8. Event by event: the 72+ h coefficient; decision rule, part 2
POOL = D[D["keep"]]
rows = []
for key in EVENTS.index:
    d = POOL[POOL["event"] == key]
    n72 = int((d["out_bin4"] == "h072p").sum())
    row = {"event": key, "name": EVENTS.loc[key, "name"], "n_72plus": n72, "held_out": key in HOLDOUT}
    if n72 >= MIN_FIT:
        m, used = fit(d, "visits_in_chg", [OUT4_T] + NOW_T + CUM_T, "ew")
        c = coefs(m, OUT4_T).set_index("level")
        if "h072p" in c.index:
            row.update(c.loc["h072p", ["coef", "se", "lo", "hi"]].to_dict())
    rows.append(row)
m_all, _ = fit(MAIN, "visits_in_chg", [OUT4_T] + NOW_T + CUM_T, "ew")
pooled = coefs(m_all, OUT4_T).set_index("level").loc["h072p"]
rows.append({"event": "pooled", "name": "Pooled (7 events)", "n_72plus": int((MAIN["out_bin4"] == "h072p").sum()), "held_out": False,
             **pooled[["coef", "se", "lo", "hi"]].to_dict()})
by_event = pd.DataFrame(rows).set_index("event")
by_event.to_csv(f"{OUT}/step2b_by_event.csv")

eligible = by_event[(by_event.index != "pooled") & ~by_event["held_out"] & (by_event["n_72plus"] >= MIN_TREATED) & by_event["coef"].notna()]
rule2 = bool(len(eligible) > 0 and (eligible["coef"] < 0).sum() > len(eligible) / 2)
print(f"Part 2: {'holds' if rule2 else 'does not hold'} ({int((eligible['coef'] < 0).sum())} of {len(eligible)} events have a negative coefficient)")
print("Conclusion:", "when outages approach or exceed three days, local activity falls by more than the storm itself can explain" if rule1 and rule2 else "in these data, the effect of outages cannot be separated from the effect of the storm itself")
tidy(by_event, 1)

## F. Event study

Another view: instead of each week's outage hours, counties are grouped by their outage duration over the whole event (Step 1b's `group`), and each week we look at how far visits in the "72+ h" and "24–72 h" groups differ from other counties. The reference week is week −2. Coefficients for weeks −5 to −3 should be close to 0 (no difference in the baseline); week −1 may show preparation and evacuation.

The weekly outage hours of the two groups are drawn too, to see whether visits or power come back first.

In [ ]:
# 9. Event study (with and without the wind and rain controls)
def es_label(g, r):
    if g not in ("ge72", "24to72") or r == ES_REF:
        return "ref"
    return f"{g}_{'m' if r < 0 else 'p'}{abs(r)}"
MAIN["es"] = [es_label(g, r) for g, r in zip(MAIN["group"], MAIN["rel_week"])]
ES_T = "C(es, Treatment('ref'))"
es_rows = []
for ctrl, terms in [("with controls", [ES_T] + NOW_T + CUM_T), ("without controls", [ES_T])]:
    m, used = fit(MAIN, "visits_in_chg", terms, "ew")
    c = coefs(m, ES_T, names={})
    c["group"] = c["level"].str.split("_").str[0]
    c["rel_week"] = c["level"].str.split("_").str[1].map(lambda s: -int(s[1:]) if s[0] == "m" else int(s[1:]))
    c["controls"] = ctrl
    es_rows.append(c)
es = pd.concat(es_rows, ignore_index=True)
for g in ["ge72", "24to72"]:
    for ctrl in es["controls"].unique():
        es = pd.concat([es, pd.DataFrame([{"group": g, "rel_week": ES_REF, "coef": 0.0, "lo": 0.0, "hi": 0.0, "controls": ctrl}])], ignore_index=True)
es = es.sort_values(["controls", "group", "rel_week"]).reset_index(drop=True)
es.to_csv(f"{OUT}/step2b_event_study.csv", index=False)
hours_prof = MAIN[MAIN["group"].isin(["ge72", "24to72"])].groupby(["group", "rel_week"])["out_week_hours"].median().unstack(0)
pre = es[(es["controls"] == "with controls") & (es["group"] == "ge72") & es["rel_week"].between(-5, -3)]
print("72+ h group, weeks −5 to −3 (should be near 0):", pre.set_index("rel_week")["coef"].round(1).to_dict())
tidy(es[es["controls"] == "with controls"].pivot(index="rel_week", columns="group", values="coef"), 1)

## G. Three figures

- **Figure 1**: the event study. Top, each group's weekly outage hours (median); bottom, the difference in visits from other counties with 95% intervals; the dashed line is the result without wind and rain controls
- **Figure 2**: the dose-response. The coefficient for each outage band, S1 (outages only) beside S3 (main), to see how much the storm itself takes up
- **Figure 3**: the 72+ h coefficient event by event. Uri, in gray, is for reference only and does not enter the decision rule

In [ ]:
# 10. Figure 1: event study
fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(9, 7.2), sharex=True, gridspec_kw={"height_ratios": [1, 2.2]})
STYLE = {"ge72": (C_DARK, "Outage of 72 h or more"), "24to72": (C_MID, "Outage of 24–72 h")}
for g, (color, label) in STYLE.items():
    if g in hours_prof:
        ax1.plot(hours_prof.index, hours_prof[g], color=color, lw=2, marker="o", ms=4, label=label)
    for ctrl, ls, alpha in [("with controls", "-", 1.0), ("without controls", "--", 0.6)]:
        s = es[(es["group"] == g) & (es["controls"] == ctrl)].sort_values("rel_week")
        off = -0.08 if g == "ge72" else 0.08
        if ctrl == "with controls":
            ax2.errorbar(s["rel_week"] + off, s["coef"], yerr=[s["coef"] - s["lo"], s["hi"] - s["coef"]], color=color, lw=2,
                         marker="o", ms=4, capsize=0, elinewidth=1.2, label=f"{label}, wind and rain controlled")
        else:
            ax2.plot(s["rel_week"] + off, s["coef"], color=color, lw=1.4, ls=ls, alpha=alpha, label=f"{label}, no controls")
for ax in (ax1, ax2):
    ax.axvline(0, color=FAINT, lw=1)
ax2.axhline(0, color=FAINT, lw=1)
ax1.set_ylabel("Hours without power\nper customer (median)")
ax1.set_title("Outage hours by week")
ax1.legend(fontsize=8.5, loc="upper right")
ax2.set_ylabel("Visits to places in the county,\nlog points vs other counties")
ax2.set_xlabel(f"Week (week 0 contains day 0; week {ES_REF} is the reference)")
ax2.set_title("Difference in visits by week, 95% intervals")
handles, labs = ax2.get_legend_handles_labels()
order = sorted(range(len(labs)), key=lambda i: ("no controls" in labs[i], "24–72" in labs[i]))
ax2.legend([handles[i] for i in order], [labs[i] for i in order], fontsize=8, loc="lower right")
ax2.set_xticks(range(-5, 7))
fig.suptitle("Event study: counties by outage duration, 7 events", x=0.01, ha="left", fontweight="bold", fontsize=12)
fig.tight_layout()
fig.savefig(f"{OUT}/step2b_fig1_event_study.png"); plt.show()

In [ ]:
# 11. Figure 2: dose-response (S1 and S3)
fig, ax = plt.subplots(figsize=(8, 4.8))
bands = [OUT_NAMES[l] for l in OUT_LEVELS[1:]]
x = np.arange(len(bands))
for s, color, off, label in [("S1", C_LIGHT, -0.12, "Outages only (S1)"), ("S3", C_DARK, 0.12, "Wind and rain controlled (S3, main)")]:
    c = main_tab[main_tab["spec"] == s].set_index("band").reindex(bands)
    ax.errorbar(x + off, c["coef"], yerr=[c["coef"] - c["lo"], c["hi"] - c["coef"]], fmt="o", color=color, ms=6,
                elinewidth=1.6, capsize=0, label=label)
n = main_tab[main_tab["spec"] == "S3"].set_index("band").reindex(bands)["n_band"]
ax.set_xticks(x)
ax.set_xticklabels([f"{b}\n(n = {int(v) if pd.notna(v) else 0})" for b, v in zip(bands, n)])
ax.axhline(0, color=FAINT, lw=1)
ax.set_xlabel("Hours without power per customer in the week (county-weeks)")
ax.set_ylabel("Visits to places in the county,\nlog points vs under 6 h")
ax.legend(fontsize=9, loc="lower left")
ax.set_title("Visits by weekly outage hours, 7 events, 95% intervals")
fig.tight_layout()
fig.savefig(f"{OUT}/step2b_fig2_dose_response.png"); plt.show()

In [ ]:
# 12. Figure 3: the 72+ h coefficient event by event
fe = by_event.dropna(subset=["coef"]).copy()
order = [k for k in EVENTS.index if k in fe.index and k not in HOLDOUT] + [k for k in HOLDOUT if k in fe.index] + ["pooled"]
fe = fe.loc[order]
fig, ax = plt.subplots(figsize=(8, 0.5 * len(fe) + 1.4))
y = np.arange(len(fe))[::-1]
for yi, (k, r) in zip(y, fe.iterrows()):
    color = C_GREY if r["held_out"] else (INK if k == "pooled" else C_DARK)
    hollow = k != "pooled" and r["n_72plus"] < MIN_TREATED                     # hollow: not counted in part 2 of the rule
    ax.errorbar(r["coef"], yi, xerr=[[r["coef"] - r["lo"]], [r["hi"] - r["coef"]]], fmt="D" if k == "pooled" else "o",
                color=color, mfc=SURFACE if hollow else color, mew=1.6, ms=6, elinewidth=1.6, capsize=0)
ax.set_yticks(y)
ax.set_yticklabels([f"{r['name']}{' (held out)' if r['held_out'] else ''}  n = {int(r['n_72plus'])}" for _, r in fe.iterrows()])
ax.axvline(0, color=FAINT, lw=1)
ax.grid(axis="y", visible=False)
ax.set_xlabel("Visits, log points: 72+ h of outage in the week vs under 6 h (95% interval)")
ax.set_title("Event by event, wind and rain controlled")
omitted = [EVENTS.loc[k, "name"] for k in EVENTS.index if k not in fe.index]
note = f"Hollow: fewer than {MIN_TREATED} county-weeks at 72+ h, not counted in the decision rule."
if omitted:
    note += f" Not shown (fewer than {MIN_FIT}): {', '.join(omitted)}."
fig.tight_layout()
fig.text(0.01, -0.02, note, ha="left", va="top", color=MUTED, fontsize=8.5)
fig.savefig(f"{OUT}/step2b_fig3_by_event.png"); plt.show()

## H. Recovery timing: do visits or power come back first?

For counties with outages of 72 hours or more, two times are compared (both in weeks; week 0 contains day 0):

- **Power-restored week**: the week in which Step 1b's `restored_time` falls (the outage share back below 10% for 24 hours)
- **Visits-recovered week**: from this week on, visits stay within 10 log points below the counties without a substantial outage

Only counties whose visits actually fell below −10 in week 0 or 1 are counted. Weekly resolution is coarse, so only "earlier, same week, later" are counted.

In [ ]:
# 13. Cross-tab of recovery timing
ref = MAIN[MAIN["group"] == "none"].groupby(["event", "rel_week"])["visits_in_chg"].median().rename("ref")
R = MAIN[MAIN["group"] == "ge72"].join(ref, on=["event", "rel_week"])
R["rel"] = R["visits_in_chg"] - R["ref"]
rows = []
for (key, f), g in R.groupby(["event", "fips"]):
    g = g.set_index("rel_week")["rel"].sort_index()
    post = g[g.index >= 0].dropna()
    if post.empty or post.loc[post.index <= 1].min() > RECOVERED:
        continue
    ok = post >= RECOVERED
    rec = next((w for w in post.index if ok.loc[w:].all()), None)
    t = L.loc[(L["event"] == key) & (L["fips"] == f), "restored_time"].iloc[0]
    if pd.isna(t):
        pw = None
    else:
        local = pd.Timestamp(t).tz_localize("UTC").tz_convert(EVENTS.loc[key, "tz"]).tz_localize(None).normalize()
        pw = int((local - pd.Timedelta(days=local.dayofweek) - EVENTS.loc[key, "week0"]).days // 7)
    if rec is None:
        verdict_ = "visits not recovered by week 6"
    elif pw is None:
        verdict_ = "power not restored in the window"
    else:
        verdict_ = "visits first" if rec < pw else ("same week" if rec == pw else "power first")
    rows.append({"event": key, "fips": f, "power_week": pw, "visits_week": rec, "order": verdict_})
rec_tab = pd.DataFrame(rows)
rec_tab.to_csv(f"{OUT}/step2b_recovery_timing.csv", index=False)
pd.crosstab(rec_tab["event"], rec_tab["order"], margins=True) if len(rec_tab) else "no counties qualify"

In [ ]:
# 14. Step 2b checklist
def verdict(passed):
    return "pending" if passed is None else ("pass" if passed else "fail")
kept = D[~D["flag_base"] & ~D["flag_prior"] & ~D["low_sample"] & D["out_week_hours"].notna()]
hz_cover = float(kept[NEEDED[1:]].notna().all(axis=1).mean())
n072, n120 = band_n(MAIN, "out_bin", "h072"), band_n(MAIN, "out_bin", "h120")
pre_max = float(pre["coef"].abs().max()) if len(pre) else np.nan
c120 = main_tab[main_tab["level"] == "h120"].set_index("spec")["coef"]
robust = bool(all(c120.get(s, np.nan) < 0 for s in ["S3", "S4", "S5"]))
checks = [
    ("Share of county-weeks with hazard data", round(hz_cover, 4), "≥ 0.99", hz_cover >= 0.99),
    ("County-weeks at 72–120 h in the main sample", n072, "≥ 30", n072 >= 30),
    ("County-weeks at 120+ h in the main sample", n120, "≥ 15", n120 >= 15),
    ("Event study: largest absolute coefficient of the 72+ h group in weeks −5 to −3", round(pre_max, 1), "≤ 5", bool(pre_max <= 5)),
    ("Decision rule part 1 (both bands significantly negative in the main spec)", str(rule1_parts), "—", rule1),
    ("Decision rule part 2 (a majority of events negative)", f"{int((eligible['coef'] < 0).sum())} / {len(eligible)}", "> 1/2", rule2),
    ("The 120+ h coefficient is negative in S3, S4 and S5", ", ".join(f"{s} {c120.get(s, np.nan):+.1f}" for s in ["S3", "S4", "S5"]), "—", robust),
]
check_tab = pd.DataFrame(checks, columns=["Indicator", "Value", "Threshold", "passed"])
check_tab["Verdict"] = check_tab.pop("passed").map(verdict)
check_tab.to_csv(f"{OUT}/step2b_checklist.csv", index=False)
check_tab